In [ ]:
from typo_utils import make_typo, ngram_overlap, evaluate_ngram_overlap_mecab, compute_micro_average, ipadic_files, compute_metrics
import MeCab
import random
import pandas as pd
from tqdm.notebook import tqdm
import typo_utils

In [ ]:
from llama_utils import get_llama_server_models
model_name = get_llama_server_models("http://localhost:8080")["data"][0]["id"]
print(model_name)

In [ ]:
import json
import re
import requests

def strip_gpt_oss_channels(s: str) -> str:
    """llama-server(gpt-oss)がcontentに混ぜる <|channel|>... を除去して本文だけ返す"""
    if not s:
        return ""

    # final チャンネルがある場合は final だけ採用
    m = re.search(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|$)", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    # analysis + <|end|> + 本文 の形式なら <|end|> 以降だけ採用
    if "<|end|>" in s:
        return s.split("<|end|>")[-1].strip()

    # それ以外：タグを雑に除去
    s = re.sub(r"<\|[^>]*\|>", "", s)
    return s.strip()

def extract_last_report(text: str) -> str:
    """テンプレが2回出る等の事故に備え、最後のヘッダ以降だけ採用"""
    if not text:
        return ""
    header = "【乳癌取扱い規約・第19版.】"
    idxs = [m.start() for m in re.finditer(re.escape(header), text)]
    if not idxs:
        return text.strip()
    return text[idxs[-1]:].strip()

def parse_llm_json(text: str) -> dict:
    # 前後の空白とクォートを除去
    text = text.strip()

    if (
        (text.startswith("'") and text.endswith("'")) or
        (text.startswith('"') and text.endswith('"'))
    ):
        text = text[1:-1]

    # ```json ... ``` を除去
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    # 念のためJSON本体だけ抽出
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError("JSON not found")

    return json.loads(m.group(0))

def fix_with_llama_server_chat(
    diagnosis: str,
    server_url: str = "http://127.0.0.1:8080/v1/chat/completions",
) -> str:
    system_prompt = """あなたは病理診断レポートから情報を抽出する編集者です。
【厳守】
- 出力は次のテンプレートに沿った文章から項目を抽出して['術前治療の有無', '術式', 'リンパ節郭清', '左右', '乳房内局在', '組織型', 'T因子', 'リンパ管侵襲', '静脈侵襲', '脂肪浸潤', '皮膚浸潤', '核異型スコア', '核分裂像スコア（核グレード分類）', '腺管形成スコア', '核分裂像スコア（組織学的グレード分類）', '核グレード', '組織学的グレード', '浸潤径', '浸潤径+乳管内進展巣']をkeyとしてJSONのみ（1回）とする
- 前置き、解説、注釈、英語、Markdown、箇条書き、コードブロック、思考過程を一切出力しない
- スコアなどは数字のみ抽出する
- 値が不明な場合は「不明」と記載する（空欄にしない）"""

    template = """【乳癌取扱い規約・第19版.】
術前治療の有無: {{術前治療の有無}}.
術式: {{術式}} + {{リンパ節郭清}}.
腫瘍の部位: {{左右}}/{{乳房内局在}}.
浸潤径: {{浸潤径}}.
浸潤径+乳管内進展巣: {{浸潤径+乳管内進展巣}}.
組織型: {{組織型}}.
T因子: {{T因子}}. リンパ管侵襲: {{リンパ管侵襲}}. 静脈侵襲: {{静脈侵襲}}.
脂肪浸潤: {{脂肪浸潤}}. 皮膚浸潤: {{皮膚浸潤}}.
核グレード分類: Grade {{核グレード}} (核異型スコア: {{核異型スコア}}点, 核分裂像スコア: {{核分裂像スコア（核グレード分類）}}点).
組織学的グレード分類: Grade {{組織学的グレード}} (腺管形成スコア: {{腺管形成スコア}}点, 核異型スコア: {{核異型スコア}}点, 核分裂像スコア: {{核分裂像スコア（組織学的グレード分類）}}点)."""

    user_prompt = f"""次の診断文はテンプレート埋め済みの病理レポートです。指定キーを抽出したJSONだけを出力してください。

テンプレート：
{template}

診断文：
{diagnosis}

JSON:"""

    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        # フォーマット厳守なら低め推奨
        "temperature": 0.1,
        #"temperature": 1.0,
        "top_p": 1.0,
        "presence_penalty": 0.0,
    }

    try:
        r = requests.post(
            server_url,
            json=payload,
            proxies={"http": None, "https": None},
            timeout=600,
        )
        r.raise_for_status()
        data = r.json()

        content = (data.get("choices", [{}])[0].get("message", {}) or {}).get("content", "")
        content = content.strip()

        # gpt-oss特有のチャンネルタグ等を除去
        content = strip_gpt_oss_channels(content)

        content = parse_llm_json(content)

        return content

    except requests.exceptions.RequestException as e:
        print("❌ LLM接続エラー:", e)
        # 可能ならレスポンス本文も出す
        try:
            print("response text:", r.text)  # type: ignore
        except Exception:
            pass
        return ""

In [ ]:
import itertools

chars = 'ABCDE'
all_patterns = []
for r in range(1, len(chars) + 1):
    perms = itertools.permutations(chars, r)
    all_patterns.extend(''.join(p) for p in perms)

def make_invasive_size():
    a = round(random.uniform(0.1, 40), 1)
    b = round(random.uniform(0.1, 40), 1)
    c = round(random.uniform(0, 100), 1)
    return f"{a}×{b}×{c}mm"

def make_entire_size(size_str):
    # 例: "12.3x24.5x8.1mm" → ['12.3', '24.5', '8.1']
    try:
        base = size_str.replace('mm', '').split('×')
        a, b, c = map(float, base)
    except Exception as e:
        raise ValueError("サイズ文字列の形式が正しくありません。'AxBxCmm' 形式で指定してください。") from e

    # 各値より大きい値をランダムに生成
    a_prime = round(random.uniform(a, max(a + 10, a + 0.2)), 1)
    b_prime = round(random.uniform(b, max(b + 10, b + 0.2)), 1)
    c_prime = round(random.uniform(c, max(c + 20, c + 0.2)), 1)

    return f"{a_prime}×{b_prime}×{c_prime}mm"

def calc_nuclear_grade(a, b):
    total = a + b
    if total<4:
        return 1
    elif total==4:
        return 2
    else:
        return 3

def calc_histological_grade(a, b, c):
    total = a + b + c
    if total<6:
        return 1
    elif total in [6, 7]:
        return 2
    else:
        return 3
    

data = {
    '術前治療の有無': ['無し', '有り'],
    '術式': ['Bp', 'Bp', 'Bt', 'Md', 'SSM', 'NSM'],
    'リンパ節郭清': ['SN→Ax(Ⅰ)', 'SN→Ax(Ⅱ)', 'SN→Ax(Ⅲ)'],
    '左右': ['左', '右', '左'],
    '乳房内局在': all_patterns,
    '組織型': [
                 'Invasive ductal carcinoma, tubule forming type',
                 'Invasive ductal carcinoma, tubule forming type',
                 'Invasive ductal carcinoma, solid type',
                 'Invasive ductal carcinoma, scirrhous type',
                 'Invasive ductal carcinoma, other type',
                 'Invasive ductal carcinoma, status post-chemotherapy',
                 'Invasive lobular carcinoma',
                 'Ductal carcinoma in situ(DCIS)',
                 'Lobular carcinoma in situ(LCIS)',
                 'Microinvasive carcinoma',
                 'Tubular carcinoma',
                 'Invasive cribriform carcinoma',
                 'Mucinous carcinoma',
                 'Medullary carcinoma',
                 'Apocrine carcinoma',
                 'Squamous cell carcinoma',
                 'Carcinoma with mesenchymal differentiation',
                 'Spindle cell carcinoma',
                 'Carcinoma with osseous/cartilaginous differentiation',
                 'Matrix-producing carcinoma',
                 'Invasive micropapillary carcinoma',
                 'Secretory carcinoma',
                 'Adenoid cystic carcinoma',
                 "Paget's disease"],
    'T因子': [
                 'pT4',
                 'pT0',
                 'pTis(DCIS)',
                 'pTis(LCIS)',
                 'pTis(Paget)',
                 'pT1mi',
                 'pT1a',
                 'pT1b',
                 'pT1c',
                 'pT2',
                 'pT3',
                 'pT4',
                 'pT4a',
                 'pT4b',
                 'pT4c',
                 'pT4d',
                 'pTX',
                 'ypT0',
                 'ypTis(DCIS)',
                 'ypTis(LCIS)',
                 'ypTis(Paget)',
                 'ypT1mi',
                 'ypT1a',
                 'ypT1b',
                 'ypT1c',
                 'ypT2',
                 'ypT3',
                 'ypT4',
                 'ypT4a',
                 'ypT4b',
                 'ypT4c',
                 'ypT4d',
                 'ypTX'],
    'リンパ管侵襲': ['Ly0', 'Ly1', 'LyX'],
    '静脈侵襲': ['V0', 'V1', 'VX'],
    '脂肪浸潤': ['f(-)', 'f(+)'],
    '皮膚浸潤': ['s(-)', 's(+)'],
    '核異型スコア': [1, 2, 3],
    '核分裂像スコア（核グレード分類）': [1, 2, 3],
    '腺管形成スコア': [1, 2, 3],
    '核分裂像スコア（組織学的グレード分類）': [1, 2, 3],
}

In [ ]:
def sample_selected():
    selected = {key: random.choice(values) for key, values in data.items()}
    grade1 = calc_nuclear_grade(selected['核異型スコア'], selected['核分裂像スコア（核グレード分類）'])
    grade2 = calc_histological_grade(selected['腺管形成スコア'], selected['核異型スコア'], selected['核分裂像スコア（組織学的グレード分類）'])
    selected['核グレード'] = grade1
    selected['組織学的グレード'] = grade2
    invasive_size = make_invasive_size()
    entire_size = make_entire_size(invasive_size)
    selected['浸潤径'] = invasive_size
    selected['浸潤径+乳管内進展巣'] = entire_size
    return selected
def make_formatted_report(selected):
    
    breast_format = f"""【乳癌取扱い規約・第19版.】
    術前治療の有無: {selected['術前治療の有無']}.
    術式: {selected['術式']} + {selected['リンパ節郭清']}.
    腫瘍の部位: {selected['左右']}/{selected['乳房内局在']}.
    浸潤径: {selected['浸潤径']}.
    浸潤径+乳管内進展巣: {selected['浸潤径+乳管内進展巣']}.
    組織型: {selected['組織型']}.
    T因子: {selected['T因子']}. リンパ管侵襲: {selected['リンパ管侵襲']}. 静脈侵襲: {selected['静脈侵襲']}.
    脂肪浸潤: {selected['脂肪浸潤']}. 皮膚浸潤: {selected['皮膚浸潤']}.
    核グレード分類: Grade {selected['核グレード']} (核異型スコア: {selected['核異型スコア']}点, 核分裂像スコア: {selected['核分裂像スコア（核グレード分類）']}点).
    組織学的グレード分類: Grade {selected['組織学的グレード']} (腺管形成スコア: {selected['腺管形成スコア']}点, 核異型スコア: {selected['核異型スコア']}点, 核分裂像スコア: {selected['核分裂像スコア（組織学的グレード分類）']}点).
    """
    return breast_format

def json_match_rate(gold: dict, pred: dict):
    keys = gold.keys()

    total = len(keys)
    match = 0
    diffs = {}

    for k in keys:
        g = str(gold.get(k, "")).strip()
        p = str(pred.get(k, "")).strip()

        if g == p:
            match += 1
        else:
            diffs[k] = {"gold": g, "pred": p}

    rate = match / total if total else 0.0

    return {
        "total_keys": total,
        "matched": match,
        "match_rate": rate,
        "diffs": diffs,
    }

In [ ]:
dic_dir = "./dictionaries/mecab-ipadic-2.7.0-20070610"
usr_dir = "./dictionaries/user.dic"

tagger = MeCab.Tagger(f'-r /dev/null -d "{dic_dir}" -u "{usr_dir}"')


In [ ]:
import time

index = list(range(100))
results_mecab_with_counts = []
results_char_with_counts = []
results_rate = []
results_match = []
results_total = []

#random.shuffle(index)
llama_times = []


for progress, i in tqdm(enumerate(index, start=1), total=len(index) ):
    selected = sample_selected()
    original = make_formatted_report(selected).strip()
    selected_str = typo_utils.normalize_unicode_symbol_white( str(selected).strip() )
    t0 = time.perf_counter()
    fixed = fix_with_llama_server_chat(original, server_url="http://localhost:8080/v1/chat/completions")
    t1 = time.perf_counter()
    llama_times.append(t1 - t0)
    fixed_str = typo_utils.normalize_unicode_symbol_white( str( fixed  ).strip() )
    

    # MeCabベースのn-gram評価
    metrics_mecab = evaluate_ngram_overlap_mecab(
        ref_text=selected_str,
        pred_text=fixed_str ,
        tagger=tagger,
        n=2,
        return_counts=True
    )
    results_mecab_with_counts.append(metrics_mecab)

    # 文字n-gram評価
    metrics_char = ngram_overlap(
        ref=selected_str,
        pred=fixed_str,
        n=3,
        return_counts=True
    )
    results_char_with_counts.append(metrics_char)

    metrics_match = json_match_rate(selected, fixed)
    results_rate += [metrics_match["match_rate"]]
    results_match += [metrics_match["matched"]] 
    results_total += [metrics_match["total_keys"]] 
    if len(metrics_match["diffs"]) > 0:
        print(metrics_match["diffs"])

    # ログ表示（100件ごと）
    if progress % 10 == 0 or progress==1:
        compute_metrics(results_mecab_with_counts, results_char_with_counts)
        
compute_metrics(results_mecab_with_counts, results_char_with_counts)
print(sum(results_rate)/len(results_match))
print(sum(results_match)/sum(results_total))

In [ ]:
import numpy as np
print(f"llama中央値応答時間: {np.median(llama_times[:100]):.3f} 秒")
print(f"llama平均応答時間: {np.mean(llama_times[:100]):.3f} 秒")
print(f"llama最小応答時間: {np.min(llama_times[:100]):.3f} 秒")
print(f"llama最大応答時間: {np.max(llama_times[:100]):.3f} 秒")